In [1]:
import torch
from torch import nn

class LayerBlock(nn.Module):
    def __init__ (self, inchannels: int, outchannels: int, time_emb_dimension , groupsize:int = 8 ):
        super().__init__()

        self.normIn = nn.GroupNorm(groupsize, inchannels)
        self.Conv1 = nn.Conv2d(inchannels, outchannels, (3,3), padding = 1)

        self.time_proj = nn.Linear(time_emb_dimension, outchannels)

        self.normOut = nn.GroupNorm(groupsize, outchannels)
        self.Conv2 = nn.Conv2d(outchannels, outchannels, (3,3), padding=1)

        self.act = nn.SiLU()

        # if channel count changes, the skip path needs a 1x1 conv to match shapes
        if inchannels != outchannels:
            self.skip = nn.Conv2d(inchannels, outchannels, kernel_size=1)
        else:
            self.skip = nn.Identity()

    def forward(self, x, t_embd):
        x_h = self.normIn(x)
        x_h = self.act(x_h)
        x_h = self.Conv1(x_h)

        # inject time information: project and broadcast-add across H, W
        time_term = self.time_proj(self.act(t_embd))
        x_h = x_h + time_term[:, :, None, None]

        x_h = self.normOut(x_h)
        x_h = self.act(x_h)
        x_h = self.Conv2(x_h)

        return x_h + self.skip(x)



In [2]:
class DownSample(nn.Module):
    def __init__ (self, channels):
        super().__init__()
        self.down = nn.Conv2d(channels, channels, (3,3), stride=2,padding = 1)

    def forward(self, x):
        x_h = self.down(x)
        return x_h

class UpSample(nn.Module):
    def __init__ (self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, (3,3),padding = 1)

    def forward(self, x):
        x_h = nn.functional.interpolate(x, scale_factor=2, mode= 'bilinear')
        return self.conv(x_h)

In [3]:
class SineTimeEmbedding(nn.Module):
    def __init__ (self,dimensions):
        super().__init__()

        if dimensions % 2 != 0:

            raise ValueError("Sinusoidal positional embedding cannot apply to odd token embedding dim (got dim={:d})".format(dimensions))

        self.half = dimensions // 2
        

    def forward(self, timestep):
        device = timestep.device
        exp = torch.arange(0,self.half, device=device).float()  / (self.half)
        freq = torch.pow(10000, exp)

        args = timestep.float()[:, None] / freq[None, :]
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)
        
        return emb

class TimeEmbedding(nn.Module):
    def __init__ (self, base_dim, high_res_dim = 512 ):
        super().__init__()
        self.sine_emd = SineTimeEmbedding(base_dim)
        self.mlp = nn.Sequential(
            nn.Linear(base_dim, high_res_dim),
            nn.SiLU(),
            nn.Linear(high_res_dim, high_res_dim)
        )

    def forward(self,timestep):
        emb = self.sine_emd(timestep)
        return self.mlp(emb)

In [4]:
class SelfAttention(nn.Module):
    def __init__ (self, channels, heads = 4, groups = 32):
        super().__init__()

        if channels % heads != 0:
            raise ValueError("Channels should be multiple of no of heads")

        self.channels = channels
        self.heads = heads

        self.norm = nn.GroupNorm(groups, channels)
        self.qkv = nn.Conv2d(channels, channels*3, kernel_size= 1)
        self.Pout = nn.Conv2d(channels, channels, kernel_size= 1)

        nn.init.zeros_(self.Pout.weight)
        nn.init.zeros_(self.Pout.bias)

    def forward(self, x_0):
        B, C, H, W = x_0.shape

        x_h = self.norm(x_0)
        qkv = self.qkv(x_h)
        q,k,v = qkv.chunk(3, dim = 1)

        def reshape(t):

            t = t.view(B , self.heads, C // self.heads, H*W)
            return t.permute(0,1,3,2)

        q = reshape(q)
        k = reshape(k)
        v = reshape(v)

        scale = (C // self.heads) ** -0.5
        attention = torch.softmax((q @ k.transpose(-2, -1))*scale, dim = -1)
        output = attention @ v

        output = output.permute(0, 1, 3, 2).reshape(B, C, H, W)
        output = self.Pout(output)
 
        return x_0 + output

In [5]:
class Encoder_level(nn.Module):
    def __init__ (self, in_ch, out_ch, img_res, time_emb_dim, Down = True):
        super().__init__()
        self.blocks = nn.ModuleList()
        self.attention = nn.ModuleList()

        use_attention = img_res in (16,8)

        ch = in_ch
        for _ in range(2):
            self.blocks.append(LayerBlock(ch, out_ch, time_emb_dim))
            self.attention.append(SelfAttention(out_ch) if use_attention else nn.Identity())
            ch = out_ch   
        self.downsample = DownSample(out_ch) if Down else None

    def forward(self, x, time_emb):
        skip_connections = []
        
        for block, attn in zip(self.blocks, self.attention):
            x = block(x, time_emb)
            x = attn(x)
            skip_connections.append(x)      # save AFTER each block, before downsampling
        if self.downsample is not None:
            x = self.downsample(x)
        return x, skip_connections


In [6]:
class Decoder_level(nn.Module):
    def __init__ (self, in_ch, out_ch, img_res, time_emb_dim,Up = True):
        super().__init__()
        self.blocks = nn.ModuleList()
        self.attention = nn.ModuleList()

        use_attention = img_res in (16,8)

        ch = in_ch
        for _ in range(2):
            self.blocks.append(LayerBlock(ch + out_ch, out_ch, time_emb_dim))
            self.attention.append(SelfAttention(out_ch) if use_attention else nn.Identity())
            ch = out_ch   

        self.upsample = UpSample(out_ch) if Up else None

    def forward(self, x, time_emb, skip_connections):

        for (block, attn) in zip(self.blocks, self.attention):
            skip_connection = skip_connections.pop()
            x = torch.cat([x , skip_connection], dim = 1)
            x = block(x, time_emb)
            x = attn(x)
                 
        if self.upsample is not None:
            x = self.upsample(x)
        return x
        


In [7]:
class Unet(nn.Module):
    def __init__ (self, 
                  in_ch = 3,                    # RGB image
                  base_ch = 128,                # channel count right after the input conv(first layer first block)
                  channel_mul = (1,2,4,4),      # channel multiplier at each resolution level
                  res_blocks = 2,               # ResBlocks per level, both encoder and decoder
                  attn = (16,8),                # apply attention only at these spatial sizes
                  time_emb_dim = 512,           # High resolution time embedding dimensions
                  image_size = 64):             #training image resolution
        
        super().__init__()

        self.Time_embedding = TimeEmbedding(base_ch, time_emb_dim)
        self.inConv = nn.Conv2d(in_ch, base_ch, kernel_size= (3,3), padding= 1)

        levels = len(channel_mul)
        channels = [base_ch * i for i in channel_mul]

        # Encoder Process
        self.Down_Levels = nn.ModuleList()
        ch = base_ch

        for i in range(levels):
            is_bottom = (i == levels - 1)
            self.Down_Levels.append(Encoder_level(ch, channels[i], image_size, time_emb_dim, Down =not is_bottom))
            ch = channels[i]
            if not is_bottom:
                image_size = image_size // 2

        # Bottelneck
        
        self.mid_block1 = LayerBlock(ch, ch, time_emb_dim)
        self.mid_attn = SelfAttention(ch)
        self.mid_block2 = LayerBlock(ch, ch, time_emb_dim)

        # Decoder Process

        self.Up_Levels = nn.ModuleList()
                
        for i in reversed(range(levels)):
            is_final = (i == 0)
            self.Up_Levels.append(Decoder_level(ch, channels[i], image_size, time_emb_dim, Up =not is_final))
            ch = channels[i]
            if not is_final:
                image_size = image_size * 2

        # ---- Output head ----
        self.output_norm = nn.GroupNorm(8, base_ch)
        self.output_act = nn.SiLU()
        self.output_conv = nn.Conv2d(base_ch, in_ch, kernel_size=3, padding=1)
        nn.init.zeros_(self.output_conv.weight)   # zero-init: network starts by predicting zero noise
        nn.init.zeros_(self.output_conv.bias)


    def forward(self, x , t):
        time_emb = self.Time_embedding(t)
        x = self.inConv(x)

        skip_connections = []

        # Encoder Forward
        for levels in self.Down_Levels:
            x , level_skips = levels(x, time_emb)
            skip_connections += level_skips

        # Bottleneck Forward
        x = self.mid_block1(x, time_emb)
        x = self.mid_attn(x)
        x = self.mid_block2(x, time_emb)

        #Decoder Forward
        for levels in (self.Up_Levels):
            x = levels(x , time_emb, skip_connections)

        x = self.output_norm(x)
        x = self.output_act(x)
        x = self.output_conv(x)

        return x
            





In [8]:
model = Unet()
x = torch.randn(2, 3, 64, 64)          # batch of 2 fake images
t = torch.randint(0, 1000, (2,))       # random timesteps
out = model(x, t)
print(out.shape)   # should print: torch.Size([2, 3, 64, 64])


torch.Size([2, 3, 64, 64])


In [9]:
# A testbench from Gemini

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Testing on device: {device}")

# 1. Instantiate the Model
model = Unet(image_size=64).to(device)

# Check total parameter count
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total Trainable Parameters: {num_params:,}")

# 2. Shape Verification Test
batch_size = 2
in_channels = 3
image_size = 64

# Dummy input tensor and timesteps
x = torch.randn(batch_size, in_channels, image_size, image_size, device=device)
t = torch.randint(0, 1000, (batch_size,), device=device)

# Forward pass
out = model(x, t)

print(f"Input shape:  {x.shape}")
print(f"Output shape: {out.shape}")

assert out.shape == x.shape, f"Shape Mismatch! Expected {x.shape}, got {out.shape}"
print("✅ Shape Test Passed!")

# 3. Backward Pass & Gradient Flow Test
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
optimizer.zero_grad()

# Dummy target noise (same shape as input)
target = torch.randn_like(out)
loss = torch.nn.functional.mse_loss(out, target)

# Backpropagate
loss.backward()

# Check gradients
missing_grads = False
nan_grads = False

for name, param in model.named_parameters():
    if param.grad is None:
        print(f"⚠️ Warning: Parameter `{name}` received no gradient!")
        missing_grads = True
    elif torch.isnan(param.grad).any():
        print(f"❌ Error: NaN gradient found in `{name}`!")
        nan_grads = True

if not missing_grads and not nan_grads:
    print("✅ Backward Pass & Gradient Flow Test Passed!")

Testing on device: cuda
Total Trainable Parameters: 87,600,899
Input shape:  torch.Size([2, 3, 64, 64])
Output shape: torch.Size([2, 3, 64, 64])
✅ Shape Test Passed!
✅ Backward Pass & Gradient Flow Test Passed!


In [10]:


def test_shape_default_small():
    # smaller base_ch/image_size for a fast CPU test, same structure as your defaults
    m = Unet(in_ch=3, base_ch=128, channel_mul=(1,2,4,4), image_size=32, time_emb_dim=128)
    x = torch.randn(2, 3, 32, 32)
    t = torch.randint(0, 1000, (2,))
    out = m(x, t)
    assert out.shape == x.shape, f"shape mismatch: {out.shape} vs {x.shape}"
    print("PASS test_shape_default_small:", out.shape)

def test_shape_true_defaults():
    # your literal defaults (heavier, still fine on CPU for batch=1)
    m = Unet()
    x = torch.randn(1, 3, 64, 64)
    t = torch.randint(0, 1000, (1,))
    out = m(x, t)
    assert out.shape == x.shape
    print("PASS test_shape_true_defaults:", out.shape)

def test_zero_init_output():
    m = Unet(in_ch=3, base_ch=128, channel_mul=(1,2,4,4), image_size=32, time_emb_dim=128)
    x = torch.randn(2, 3, 32, 32)
    t = torch.randint(0, 1000, (2,))
    out = m(x, t)
    assert torch.allclose(out, torch.zeros_like(out)), "output should be exactly zero at init (zero-init output conv)"
    print("PASS test_zero_init_output: output is all zeros at init, as expected")

def test_backward_all_params_get_grad():
    m = Unet(in_ch=3, base_ch=128, channel_mul=(1,2,4,4), image_size=32, time_emb_dim=128)
    x = torch.randn(2, 3, 32, 32, requires_grad=True)
    t = torch.randint(0, 1000, (2,))
    out = m(x, t)
    # output is zero everywhere at init, so grad w.r.t. output_conv won't reach earlier
    # layers through that path; use a loss that also pulls on internal activations
    # by adding an epsilon perturbation via a hook-free trick: sum of squares of a
    # mid-network tensor is not directly accessible, so instead disable zero-init
    # for this specific gradient-flow check.
    nn.init.normal_(m.output_conv.weight, std=0.02)
    out = m(x, t)
    loss = out.pow(2).mean()
    loss.backward()
    missing = [n for n, p in m.named_parameters() if p.requires_grad and p.grad is None]
    assert not missing, f"parameters with no gradient: {missing}"
    assert x.grad is not None, "no gradient reached the input"
    print("PASS test_backward_all_params_get_grad: all params received gradients")

def test_various_batch_sizes():
    m = Unet(in_ch=3, base_ch=128, channel_mul=(1,2,4,4), image_size=32, time_emb_dim=128)
    for b in (1, 3, 5):
        x = torch.randn(b, 3, 32, 32)
        t = torch.randint(0, 1000, (b,))
        out = m(x, t)
        assert out.shape == (b, 3, 32, 32)
    print("PASS test_various_batch_sizes")

def test_different_level_counts():
    # 3-level and 5-level configs to make sure skip-connection bookkeeping generalizes
    for channel_mul in [(1,2,4), (1,1,2,2,4)]:
        m = Unet(in_ch=3, base_ch=128, channel_mul=channel_mul, image_size=32, time_emb_dim=64)
        x = torch.randn(2, 3, 32, 32)
        t = torch.randint(0, 1000, (2,))
        out = m(x, t)
        assert out.shape == x.shape, f"failed for channel_mul={channel_mul}, got {out.shape}"
    print("PASS test_different_level_counts")

def test_skip_connection_count_matches():
    # Directly check encoder produces exactly as many skips as decoder consumes
    m = Unet(in_ch=3, base_ch=128, channel_mul=(1,2,4,4), image_size=32, time_emb_dim=64)
    x = torch.randn(1, 3, 32, 32)
    time_emb = m.Time_embedding(torch.randint(0, 1000, (1,)))
    x = m.inConv(x)
    skips = []
    for level in m.Down_Levels:
        x, level_skips = level(x, time_emb)
        skips += level_skips
    produced = len(skips)
    x = m.mid_block1(x, time_emb); x = m.mid_attn(x); x = m.mid_block2(x, time_emb)
    consumed_start = len(skips)
    for level in m.Up_Levels:
        x = level(x, time_emb, skips)
    consumed = consumed_start - len(skips)
    assert produced == consumed, f"produced {produced} skips but consumed {consumed}"
    assert len(skips) == 0, f"skip list not fully drained, {len(skips)} left over"
    print(f"PASS test_skip_connection_count_matches: {produced} produced == {consumed} consumed")

def test_attn_parameter_is_ignored_BUG():
    """
    Demonstrates that Unet(attn=...) has NO effect on attention placement:
    Encoder_level / Decoder_level hardcode `img_res in (16, 8)` internally
    instead of receiving the `attn` tuple from Unet. Proof: two models built
    with very different `attn` values should have DIFFERENT attention-module
    placement if `attn` were respected -- but they come out identical.
    This test documents the current (buggy) behavior; it will start FAILING
    once you fix the bug by threading `attn` through (at which point
    delete/update this test).
    """
    kwargs = dict(in_ch=3, base_ch=128, channel_mul=(1,2,4,4), image_size=32, time_emb_dim=64)
    m1 = Unet(attn=(32,), **kwargs)          # asks for attention at res 32 only
    m2 = Unet(attn=(), **kwargs)             # asks for NO attention anywhere

    def attn_pattern(m):
        def has_real_attention(level):
            return [not isinstance(a, nn.Identity) for a in level.attention]
        return [has_real_attention(l) for l in m.Down_Levels] + [has_real_attention(l) for l in m.Up_Levels]

    p1, p2 = attn_pattern(m1), attn_pattern(m2)
    print("attn=(32,) pattern:", p1)
    print("attn=()    pattern:", p2)

    # If `attn` were respected these would differ (p2 should be all-False).
    # Currently they're identical, proving the argument is ignored.
    assert p1 == p2, "If this assertion fails, the bug has been fixed (attn is now respected) -- great!"
    print("CONFIRMED BUG: `attn` constructor argument is currently ignored; "
          "attention placement is hardcoded to resolutions (16, 8) inside "
          "Encoder_level/Decoder_level regardless of what you pass to Unet(attn=...).")

def test_groupnorm_divisibility_guard():
    # base_ch not divisible by 8 -> output_norm = GroupNorm(8, base_ch) should raise
    try:
        Unet(in_ch=3, base_ch=10, channel_mul=(1,2), image_size=16, time_emb_dim=32)
        raised = False
    except ValueError:
        raised = True
    assert raised, "expected GroupNorm(8, base_ch) to raise when base_ch % 8 != 0"
    print("PASS test_groupnorm_divisibility_guard: confirms base_ch must be divisible by 8")

def test_param_count():
    m = Unet(in_ch=3, base_ch=128, channel_mul=(1,2,4,4), image_size=32, time_emb_dim=128)
    n_params = sum(p.numel() for p in m.parameters())
    print(f"INFO param count (base_ch=128 config): {n_params:,}")

if __name__ == "__main__":
    tests = [
        test_shape_default_small,
        test_shape_true_defaults,
        test_zero_init_output,
        test_backward_all_params_get_grad,
        test_various_batch_sizes,
        test_different_level_counts,
        test_skip_connection_count_matches,
        test_attn_parameter_is_ignored_BUG,
        test_groupnorm_divisibility_guard,
        test_param_count,
    ]
    failures = []
    for t in tests:
        try:
            t()
        except Exception as e:
            failures.append((t.__name__, e))
            print(f"FAIL {t.__name__}: {e}")
    print("\n==== SUMMARY ====")
    print(f"{len(tests) - len(failures)}/{len(tests)} passed")
    if failures:
        for name, e in failures:
            print(f" - {name}: {e}")

PASS test_shape_default_small: torch.Size([2, 3, 32, 32])
PASS test_shape_true_defaults: torch.Size([1, 3, 64, 64])
PASS test_zero_init_output: output is all zeros at init, as expected
PASS test_backward_all_params_get_grad: all params received gradients
PASS test_various_batch_sizes
PASS test_different_level_counts
PASS test_skip_connection_count_matches: 8 produced == 8 consumed
attn=(32,) pattern: [[False, False], [True, True], [True, True], [False, False], [False, False], [True, True], [True, True], [False, False]]
attn=()    pattern: [[False, False], [True, True], [True, True], [False, False], [False, False], [True, True], [True, True], [False, False]]
CONFIRMED BUG: `attn` constructor argument is currently ignored; attention placement is hardcoded to resolutions (16, 8) inside Encoder_level/Decoder_level regardless of what you pass to Unet(attn=...).
PASS test_groupnorm_divisibility_guard: confirms base_ch must be divisible by 8
INFO param count (base_ch=128 config): 81,597,443

